In [1]:
from py2neo import Node, Relationship, Graph, Subgraph
import pandas as pd
from tqdm import tqdm
import re
import numpy as np

# Connect to neo4j database

In [2]:
# with neo4j running
# graph = Graph("bolt://localhost:7687", auth=("neo4j", "igem2024")) # TO BE MODIFIED
# graph.delete_all()

# Load data

In [3]:
def gradient_color_generate(time: str):
    from datetime import datetime
    import matplotlib as mpl
    begin_time = datetime.strptime('2004-1-1', r'%Y-%m-%d')
    try:
        exact_time = datetime.strptime(time, r'%Y-%m-%d')
    except:
        exact_time = datetime.strptime('2004-1-1', r'%Y-%m-%d')
    now = datetime.now()
    max_interavl = now - begin_time
    interval = now - exact_time
    norm = mpl.colors.Normalize(vmin=0, vmax=max_interavl.days)
    cmap = mpl.colormaps.get_cmap('plasma')
    hex_rgb = mpl.colors.rgb2hex(cmap(norm(interval.days)))
    return hex_rgb

In [4]:
text_embedding_mp = {}
text_embedding_list = []
text_num_list = []
with open('../similarity/data/text_embeddings.csv', 'r') as f: # TO BE MODIFIED
    for i in range(70000): # TO BE MODIFIED
        line = f.readline()[:-1]
        if not line:
            break
        part_num, text_embedding = line.split(',')
        text_embedding = list(map(float, text_embedding.split(' ')))
        text_num_list.append(part_num)
        text_embedding_list.append(text_embedding)
        text_embedding_mp.update({part_num: text_embedding})

In [23]:
ref_twins_edges = set()
part_node_dict = {}
for yr in range(2004, 2024):
    print(f'Uploading {yr}...', flush=True)
    data = pd.read_csv(f'../parthub/collections/{yr}collection.csv')
    part_list = []
    relationship_list = []
    for i in data.index:
        part_num = str(data['part_num'].values[i])
        part_name = str(data['part_name'].values[i])
        part_url = str(data['part_url'].values[i])
        part_desc = str(data['short_desc'].values[i])
        part_type = str(data['part_type'].values[i])
        part_team = str(data['team'].values[i])
        part_sequence = str(data['sequence'].values[i])
        part_contents = str(re.sub(' Sequence and Features', '', str(data['contents'].values[i])))
        part_released = str(data['released'].values[i])
        part_sample = str(data['sample'].values[i])
        part_twins = str(data['twins'].values[i])
        part_assemble = str(data['assemble_std'].values[i])
        part_used = str(data['parts_used'].values[i])
        part_using = str(data['using_parts'].values[i])
        part_len = str(data['len'].values[i])
        part_date = str(data['date'].values[i])
        part_isfavorite = str(data['isfavorite'].values[i])
        part_year = str(data['year'].values[i])
        part_designer = str(data['designer'].values[i])
        try:
            part_used_list = part_used.split(' ')
            part_using_list = part_using.split(' ')
            part_twins_list = part_twins.split(' ')
            if part_used == 'None' or part_used == '' or part_used == 'N o n e':
                part_used_list = []
            if part_using == 'self' or part_using == '':
                part_using_list = []
            if part_twins == 'None' or part_twins == '' or part_twins == 'N o n e' or part_twins.lower() == 'nan':
                part_twins_list = []
        except:
            part_used_list = []
            part_using_list = []
            part_twins_list = []
        try:
            text_embedding = text_embedding_mp[str(part_num)]
        except:
            text_embedding = [0] * 768
        part_node = Node('Part', number=str(part_num), name=part_name, url=part_url, description=part_desc, type=part_type,
                        team=part_team, sequence=part_sequence, contents=part_contents, released=part_released,
                        sample=part_sample, assemble=part_assemble, length=part_len, date=part_date,
                        isfavorite=str(part_isfavorite), twins=part_twins_list, twins_num=str(len(part_twins_list)),
                        cited_by=part_used_list, year=part_year, cites=str(len(part_used_list)), ref=part_using_list,
                        citing=str(len(part_using_list)), designer=part_designer, prweight=max(1,len(part_used_list) * 0.5+len(part_using_list)+0.75 * len(part_twins_list)),
                        color=gradient_color_generate(part_date), textEmbedding=text_embedding)
        part_list.append(part_node)
        part_node_dict.update({str(part_num): part_node})
    twins_set = set()
    for pNode in part_list:
        if pNode['ref']:
            for ref_part in pNode['ref']:
                try:
                    pNode1 = part_node_dict[ref_part]
                    relationShip = Relationship(pNode, 'refers to', pNode1)
                    relationShip["weight"] = pNode['prweight']
                    relationship_list.append(relationShip)
                except:
                    pass
        if pNode['twins']:
            for twin_part in pNode['twins']:
                try:
                    pNode2 = part_node_dict[twin_part]
                    if pNode2['number'] != pNode['number']:
                        if set([pNode2['number'],pNode['number']]) not in twins_set:
                            relationShip = Relationship(pNode, 'twins', pNode2)
                            relationship_list.append(relationShip)
                            twins_set.add(set([pNode2['number'],pNode['number']]))
                except:
                    pass
        if pNode['cited_by']:
            for cite_part in pNode['cited_by']:
                try:
                    pNode3 = part_node_dict[cite_part]
                    relationShip = Relationship(pNode3, 'refers to', pNode)
                    relationShip["weight"] = pNode3['prweight']
                    relationship_list.append(relationShip)
                except:
                    pass
    relationship_list = list(set(relationship_list))
    for relationship in relationship_list:
        ref_twins_edges.add(relationship.nodes)

Uploading 2004...
Uploading 2005...
Uploading 2006...
Uploading 2007...
Uploading 2008...
Uploading 2009...
Uploading 2010...
Uploading 2011...
Uploading 2012...
Uploading 2013...
Uploading 2014...
Uploading 2015...
Uploading 2016...
Uploading 2017...
Uploading 2018...
Uploading 2019...
Uploading 2020...
Uploading 2021...
Uploading 2022...
Uploading 2023...


In [8]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
knn.fit(text_embedding_list)

NearestNeighbors(metric='cosine')

In [20]:
dist = knn.kneighbors_graph(text_embedding_list, n_neighbors=5, mode='distance')

<66282x66282 sparse matrix of type '<class 'numpy.float64'>'
	with 331410 stored elements in Compressed Sparse Row format>

In [ ]:
cnt=0
for row in range(dist.shape[0]):
    for col in dist.indices[dist.indptr[row]:dist.indptr[row+1]]:
        cnt+=1
        print(f"value at row {text_num_list[row]} and column {text_num_list[col]} is {1 - dist[row, col]}")
        if cnt>30: break
    if cnt>30: break

In [26]:
relationship_list = []
for row in range(dist.shape[0]):
    for col in dist.indices[dist.indptr[row]:dist.indptr[row+1]]:
        if (row, col) not in ref_twins_edges and (col, row) not in ref_twins_edges:
            pNode1 = part_node_dict[text_num_list[row]]
            pNode2 = part_node_dict[text_num_list[col]]
            relationShip = Relationship(pNode1, 'text_similar', pNode2, score=dist[row, col])
            relationship_list.append(relationShip)
            ref_twins_edges.add((row, col))

# Upload to neo4j

In [ ]:
subgraph = Subgraph(part_list, relationship_list)
tx = graph.begin()
tx.create(subgraph)
graph.commit(tx)

# calculate PageRank

In [ ]:
# create graph
query = """
CALL gds.graph.project(
'parthub',
'Part',
'refers to',
{
    nodeProperties: ['textEmbedding'],
    relationshipProperties: 'weight'
}
)
"""
graph.run(query)

In [ ]:
# calculate PageRanks
query = '''
CALL gds.pageRank.write('parthub', {
  maxIterations: 20,
  dampingFactor: 0.85,
  writeProperty: 'pagerank',
  relationshipWeightProperty: 'weight'
})
YIELD nodePropertiesWritten, ranIterations
'''
graph.run(query)

In [ ]:
# get max pagerank and min pagerank
query = '''
MATCH (n:Part)
RETURN max(n.pagerank) AS max_val, min(n.pagerank) AS min_val
'''
graph.run(query)

In [ ]:
# generate nodesize
query = '''
MATCH (n:Part)
SET n.nodesize = (n.pagerank - 0.15000000000000002) / (46.258541601845714 - 0.15000000000000002) * 90 + 30
'''
graph.run(query)

# Louvain method

In [ ]:
# run Louvain method
query = '''
CALL gds.louvain.write('parthub', {
  writeProperty: 'community',
  relationshipWeightProperty: 'weight'
})
'''
graph.run(query)

# KNN

In [ ]:
# Calculate KNN
# query = '''
# CALL gds.knn.write('parthub', {
#     writeRelationshipType: 'SIMILAR',
#     writeProperty: 'score',
#     topK: 5,
#     randomSeed: 233,
#     concurrency: 1,
#     nodeProperties: ['textEmbedding']
# })
# YIELD nodesCompared, relationshipsWritten
# '''
# graph.run(query)

# delete temp graph

In [ ]:
query = '''
CALL gds.graph.drop('parthub')
'''
graph.run(query)